In [1]:
import modified_didppy as dp
import vrplib
import numpy as np
import math
import tempfile
import os

# **Data**

## Small-sized testing data

In [2]:
# =====================================================================================
# 1. Data and Preprocessing
# =====================================================================================
# Number of locations (0 is depot)
num_locations = 4
# Number of vehicles
num_vehicles = 2
# Vehicle capacity
q = 5
# Demand of each customer
cust_demand = [0, 2, 3, 3]
# Ready time (earliest arrival)
avail_time = [0, 5, 0, 8]
# Deadline (latest arrival)
due_date = [100, 16, 10, 14]
# Service time
serve_time = [0, 0, 0, 0]
# Travel cost matrix
travel_cost = [[0, 3, 4, 5], [3, 0, 5, 4], [4, 5, 0, 3], [5, 4, 3, 0]]

## Solomon C101 dataset

In [13]:
C101_txt ="""
C101

VEHICLE
NUMBER     CAPACITY
  25         200

CUSTOMER
CUST NO.  XCOORD.   YCOORD.    DEMAND   READY TIME  DUE DATE   SERVICE   TIME

    0      40         50          0          0       1236          0
    1      45         68         10        912        967         90
    2      45         70         30        825        870         90
    3      42         66         10         65        146         90
    4      42         68         10        727        782         90
    5      42         65         10         15         67         90
    6      40         69         20        621        702         90
    7      40         66         20        170        225         90
    8      38         68         20        255        324         90
    9      38         70         10        534        605         90
   10      35         66         10        357        410         90
   11      35         69         10        448        505         90
   12      25         85         20        652        721         90
   13      22         75         30         30         92         90
   14      22         85         10        567        620         90
   15      20         80         40        384        429         90
   16      20         85         40        475        528         90
   17      18         75         20         99        148         90
   18      15         75         20        179        254         90
   19      15         80         10        278        345         90
   20      30         50         10         10         73         90
   21      30         52         20        914        965         90
   22      28         52         20        812        883         90
   23      28         55         10        732        777         90
   24      25         50         10         65        144         90
   25      25         52         40        169        224         90
   26      25         55         10        622        701         90
   27      23         52         10        261        316         90
   28      23         55         20        546        593         90
   29      20         50         10        358        405         90
   30      20         55         10        449        504         90
   31      10         35         20        200        237         90
   32      10         40         30         31        100         90
   33       8         40         40         87        158         90
   34       8         45         20        751        816         90
   35       5         35         10        283        344         90
   36       5         45         10        665        716         90
   37       2         40         20        383        434         90
   38       0         40         30        479        522         90
   39       0         45         20        567        624         90
   40      35         30         10        264        321         90
   41      35         32         10        166        235         90
   42      33         32         20         68        149         90
   43      33         35         10         16         80         90
   44      32         30         10        359        412         90
   45      30         30         10        541        600         90
   46      30         32         30        448        509         90
   47      30         35         10       1054       1127         90
   48      28         30         10        632        693         90
   49      28         35         10       1001       1066         90
   50      26         32         10        815        880         90
   51      25         30         10        725        786         90
   52      25         35         10        912        969         90
   53      44          5         20        286        347         90
   54      42         10         40        186        257         90
   55      42         15         10         95        158         90
   56      40          5         30        385        436         90
   57      40         15         40         35         87         90
   58      38          5         30        471        534         90
   59      38         15         10        651        740         90
   60      35          5         20        562        629         90
   61      50         30         10        531        610         90
   62      50         35         20        262        317         90
   63      50         40         50        171        218         90
   64      48         30         10        632        693         90
   65      48         40         10         76        129         90
   66      47         35         10        826        875         90
   67      47         40         10         12         77         90
   68      45         30         10        734        777         90
   69      45         35         10        916        969         90
   70      95         30         30        387        456         90
   71      95         35         20        293        360         90
   72      53         30         10        450        505         90
   73      92         30         10        478        551         90
   74      53         35         50        353        412         90
   75      45         65         20        997       1068         90
   76      90         35         10        203        260         90
   77      88         30         10        574        643         90
   78      88         35         20        109        170         90
   79      87         30         10        668        731         90
   80      85         25         10        769        820         90
   81      85         35         30         47        124         90
   82      75         55         20        369        420         90
   83      72         55         10        265        338         90
   84      70         58         20        458        523         90
   85      68         60         30        555        612         90
   86      66         55         10        173        238         90
   87      65         55         20         85        144         90
   88      65         60         30        645        708         90
   89      63         58         10        737        802         90
   90      60         55         10         20         84         90
   91      60         60         10        836        889         90
   92      67         85         20        368        441         90
   93      65         85         40        475        518         90
   94      65         82         10        285        336         90
   95      62         80         30        196        239         90
   96      60         80         10         95        156         90
   97      60         85         30        561        622         90
   98      58         75         20         30         84         90
   99      55         80         10        743        820         90
  100      55         85         20        647        726         90
"""

In [14]:
C101_sol = """
Route #1: 5 3 7 8 10 11 9 6 4 2 1 75
Route #2: 13 17 18 19 15 16 14 12
Route #3: 20 24 25 27 29 30 28 26 23 22 21
Route #4: 32 33 31 35 37 38 39 36 34
Route #5: 43 42 41 40 44 46 45 48 51 50 52 49 47
Route #6: 57 55 54 53 56 58 60 59
Route #7: 67 65 63 62 74 72 61 64 68 66 69
Route #8: 81 78 76 71 70 73 77 79 80
Route #9: 90 87 86 83 82 84 85 88 89 91
Route #10: 98 96 95 94 92 93 97 100 99
Cost 827.3
"""

### Reading Solomon Data

In [15]:
# Create a temporary file to store the C101_txt content
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as temp_file:
    temp_file.write(C101_txt)
    temp_file_path = temp_file.name

try:
    solomon_data = vrplib.read_instance(temp_file_path, instance_format='solomon')
    #display(solomon_data)
finally:
    # Clean up the temporary file
    os.remove(temp_file_path)


In [16]:
num_locations = len(solomon_data['node_coord'])
num_vehicles = solomon_data['vehicles']
q = float(solomon_data['capacity'])
cust_demand = solomon_data['demand'].tolist()
cust_demand = [float(d) for d in cust_demand]
avail_time = solomon_data['time_window'][:, 0].tolist()
avail_time = [float(d) for d in avail_time]
due_date = solomon_data['time_window'][:, 1].tolist()
due_date = [float(d) for d in due_date]
serve_time= solomon_data['service_time'].tolist()
serve_time = [float(d) for d in serve_time]

print(f"Number of locations : {num_locations}")
print(f"Number of vehicles : {num_vehicles}")
print(f"Vehicle capacity (q): {q}")
print(f"Demand : {cust_demand}")
print(f"Ready time: {avail_time}")
print(f"Due date (b): {due_date}")
print(f"Service time (s): {serve_time}")

# Extract coordinates for distance calculation
coords = solomon_data['node_coord']

# Initialize the cost matrix
travel_cost = np.zeros((num_locations, num_locations))

# Calculate Euclidean distances
for i in range(num_locations):
    for j in range(num_locations):
        if i == j:
            travel_cost[i, j] = 0
        else:
            travel_cost[i, j] = math.dist(coords[i], coords[j])

# Convert to list of lists (if DIDPPY expects this format) and round to nearest integer as typically done in VRP
travel_cost = [[float(val) for val in row] for row in travel_cost]

print(f"\nTravel cost matrix (c):\n{np.array(travel_cost)}")

Number of locations : 101
Number of vehicles : 25
Vehicle capacity (q): 200.0
Demand : [0.0, 10.0, 30.0, 10.0, 10.0, 10.0, 20.0, 20.0, 20.0, 10.0, 10.0, 10.0, 20.0, 30.0, 10.0, 40.0, 40.0, 20.0, 20.0, 10.0, 10.0, 20.0, 20.0, 10.0, 10.0, 40.0, 10.0, 10.0, 20.0, 10.0, 10.0, 20.0, 30.0, 40.0, 20.0, 10.0, 10.0, 20.0, 30.0, 20.0, 10.0, 10.0, 20.0, 10.0, 10.0, 10.0, 30.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 20.0, 40.0, 10.0, 30.0, 40.0, 30.0, 10.0, 20.0, 10.0, 20.0, 50.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 30.0, 20.0, 10.0, 10.0, 50.0, 20.0, 10.0, 10.0, 20.0, 10.0, 10.0, 30.0, 20.0, 10.0, 20.0, 30.0, 10.0, 20.0, 30.0, 10.0, 10.0, 10.0, 20.0, 40.0, 10.0, 30.0, 10.0, 30.0, 20.0, 10.0, 20.0]
Ready time: [0.0, 912.0, 825.0, 65.0, 727.0, 15.0, 621.0, 170.0, 255.0, 534.0, 357.0, 448.0, 652.0, 30.0, 567.0, 384.0, 475.0, 99.0, 179.0, 278.0, 10.0, 914.0, 812.0, 732.0, 65.0, 169.0, 622.0, 261.0, 546.0, 358.0, 449.0, 200.0, 31.0, 87.0, 751.0, 283.0, 665.0, 383.0, 479.0, 567.0, 264.0, 166.0, 68.0, 16.0

In [17]:
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as temp_sol_file:
    temp_sol_file.write(C101_sol)
    temp_sol_file_path = temp_sol_file.name

try:
    solomon_solution = vrplib.read_solution(temp_sol_file_path)
    print("Solution Data:")
    display(solomon_solution)
finally:
    os.remove(temp_sol_file_path)

Solution Data:


{'routes': [[5, 3, 7, 8, 10, 11, 9, 6, 4, 2, 1, 75],
  [13, 17, 18, 19, 15, 16, 14, 12],
  [20, 24, 25, 27, 29, 30, 28, 26, 23, 22, 21],
  [32, 33, 31, 35, 37, 38, 39, 36, 34],
  [43, 42, 41, 40, 44, 46, 45, 48, 51, 50, 52, 49, 47],
  [57, 55, 54, 53, 56, 58, 60, 59],
  [67, 65, 63, 62, 74, 72, 61, 64, 68, 66, 69],
  [81, 78, 76, 71, 70, 73, 77, 79, 80],
  [90, 87, 86, 83, 82, 84, 85, 88, 89, 91],
  [98, 96, 95, 94, 92, 93, 97, 100, 99]],
 'cost': 827.3}

# **DIDP 2-transition model**

In [ ]:
# =====================================================================================
# 2. DIDP Model Definition
# =====================================================================================
model = dp.Model(float_cost=True)

# Object types for customers/locations and vehicles
customer = model.add_object_type(number=num_locations)
vehicle = model.add_object_type(number=num_vehicles)

# -------------------- State Variables --------------------
# Set of unvisited customers
unvisited_locations = model.add_set_var(object_type=customer, target=list(range(1, num_locations)))

# Per-vehicle state variables, stored in Python lists for easy access
vehicle_locations = [
  model.add_element_var(object_type=customer, target=0, name=f"loc_v{v}")
  for v in range(num_vehicles)
]
vehicle_loads = [
  model.add_float_var(target=0, name=f"load_v{v}")
  for v in range(num_vehicles)
]
vehicle_times = [
  model.add_float_resource_var(target=0, less_is_better=True, name=f"time_v{v}")
  for v in range(num_vehicles)
]
chosen_customer = model.add_element_var(object_type=customer, target=0, name="chosen_customer")

alpha = model.add_int_var(target=0, name="alpha")

# -------------------- Tables of Constants --------------------
demand = model.add_float_table(cust_demand)
ready_time = model.add_float_table(avail_time)
due_time = model.add_float_table(due_date)
service_time = model.add_float_table(serve_time)
travel_time = model.add_float_table(travel_cost)

# -------------------- Transitions --------------------
# Choose customer j to be visited next
for j in range(1, num_locations):
  choosing_customer_transition = dp.Transition(
      name=f"choose_customer_{j}_to_visit",
      cost=dp.FloatExpr.state_cost(),
      preconditions =[
          unvisited_locations.contains(j),
           alpha == 0
           ],
      effects=[
          (chosen_customer, j),
           (alpha, 1)
           ],
  )
  model.add_transition(choosing_customer_transition)

# Transition to visit a customer j with a vehicle v
for v in range(num_vehicles):
  arrival_time = dp.max(
      vehicle_times[v] + travel_time[vehicle_locations[v], chosen_customer],
      ready_time[chosen_customer]
  )

  departure_time = arrival_time + service_time[chosen_customer]

  visit_transition = dp.Transition(
      name=f"visit_chosen_customer_with_vehicle_{v}",
      cost=travel_time[vehicle_locations[v], chosen_customer] + dp.FloatExpr.state_cost(),
      preconditions=[
          unvisited_locations.contains(chosen_customer),
          vehicle_loads[v] + demand[chosen_customer] <= q,
          arrival_time <= due_time[chosen_customer],
          alpha == 1,
      ],
      effects=[
          (unvisited_locations, unvisited_locations.remove(chosen_customer)),
          (vehicle_locations[v], chosen_customer),
          (vehicle_loads[v], vehicle_loads[v] + demand[chosen_customer]),
          (vehicle_times[v], departure_time),
          (alpha, 0),
      ],
  )

  model.add_transition(visit_transition)


# Transitions for each vehicle to return to the depot after all customers are served
for v in range(num_vehicles):
  return_to_depot_transition = dp.Transition(
      name=f"return_vehicle_{v}_to_depot",
      cost=travel_time[vehicle_locations[v], 0] + dp.FloatExpr.state_cost(),
      preconditions=[unvisited_locations.is_empty(), vehicle_locations[v] != 0],
      effects=[(vehicle_locations[v], 0)],
  )
  model.add_transition(return_to_depot_transition)

# -------------------- Base Case --------------------
# All customers visited AND all vehicles are at the depot
base_conditions = [unvisited_locations.is_empty()]
for v in range(num_vehicles):
  base_conditions.append(vehicle_locations[v] == 0)
model.add_base_case(base_conditions)

# -------------------- State Constraints and Dual Bounds --------------------
# State constraint: if a customer is unvisited, it must be reachable in time by at least one vehicle
# This is complex to model perfectly, so we rely on transition preconditions for feasibility.
# A simpler constraint is that remaining capacity must meet remaining demand.
total_capacity_left = sum([(q - load) for load in vehicle_loads])
model.add_state_constr(total_capacity_left >= demand[unvisited_locations])

# Dual bound: sum of minimum travel costs to remaining customers
min_to = model.add_float_table(
  [min(travel_cost[k][j] for k in range(num_locations) if k != j) if j != 0 else 0 for j in range(num_locations)]
)
model.add_dual_bound(min_to[unvisited_locations])
min_from = model.add_float_table(
  [min(travel_cost[j][k] for k in range(num_locations) if k != j) if j != 0 else 0 for j in range(num_locations)]
)
model.add_dual_bound(min_from[unvisited_locations])



In [19]:
# =====================================================================================
# 3. Solving the Model
# =====================================================================================
print("Solving CVRPTW model...")
# Using CABS solver, which is a good general-purpose choice
solver = dp.CABS(model, quiet=True, time_limit = 10)
solution = solver.search()

if solution.is_infeasible:
  print("The problem is infeasible.")
else:
  print("\nSolution Found:")
  print("Transitions to apply:")
  for t in solution.transitions:
      print(f"- {t.name}")
  print(f"\nOptimal Cost: {solution.cost}")
  print(f"Is optimal: {solution.is_optimal}")


Solving CVRPTW model...

Solution Found:
Transitions to apply:
- choose_customer_7_to_visit
- visit_chosen_customer_with_vehicle_0
- choose_customer_6_to_visit
- visit_chosen_customer_with_vehicle_0
- choose_customer_4_to_visit
- visit_chosen_customer_with_vehicle_0
- choose_customer_1_to_visit
- visit_chosen_customer_with_vehicle_0
- choose_customer_10_to_visit
- visit_chosen_customer_with_vehicle_1
- choose_customer_11_to_visit
- visit_chosen_customer_with_vehicle_1
- choose_customer_9_to_visit
- visit_chosen_customer_with_vehicle_1
- choose_customer_2_to_visit
- visit_chosen_customer_with_vehicle_1
- choose_customer_5_to_visit
- visit_chosen_customer_with_vehicle_2
- choose_customer_3_to_visit
- visit_chosen_customer_with_vehicle_2
- choose_customer_8_to_visit
- visit_chosen_customer_with_vehicle_2
- choose_customer_15_to_visit
- visit_chosen_customer_with_vehicle_2
- choose_customer_16_to_visit
- visit_chosen_customer_with_vehicle_2
- choose_customer_14_to_visit
- visit_chosen_cust